# Get to Know a Dataset: Haploblocks (haploblocks.org)

This notebook serves as a guided tour of the [Haploblocks](https://registry.opendata.aws/[REGISTRY_YAML_NAME]) dataset — genome-wide haplotype block boundaries, sequence clusters, and per-individual genomic hashes derived from population-scale whole-genome sequencing data. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The methodology behind this dataset is described in [Kubica et al., 2025, BioHackrXiv](https://github.com/collaborativebioinformatics/Haploblock_Clusters_ElixirBH25): the genome is divided into haploblocks (regions between recombination hotspots), and individuals' genomic sequences within each haploblock are clustered by similarity and encoded into compact binary hashes. This turns population-scale variant data into a discretized, per-region representation that's fast to query and compare across individuals and populations.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket is organized per-chromosome. Within each chromosome prefix (`chr1/` ... `chr22/`, `chrX/`), you'll find:

1. **Per-haploblock-region files**, named by their genomic coordinates (`START-END`):
   - `chrN_START-END_all_seqs.fasta` — all haplotype sequences in that region
   - `chrN_START-END_rep_seq.fasta` — one representative sequence per cluster
   - `chrN_START-END_cluster.tsv` — pairwise mapping from each cluster representative to its members (no header)
2. **Genome-wide summary files** (one per chromosome):
   - `haploblock_boundaries_chrN.tsv` — START/END coordinates of every haploblock
   - `haploblock_hashes.tsv` — START/END/HASH, the region-level hash identifying each haploblock
   - `individual_hashes_START-END.tsv` — INDIVIDUAL/HASH, one row per haplotype (`_hap0`/`_hap1`) per sample, giving each individual's cluster assignment in that region
   - `cluster_hashes_START-END.tsv` — CLUSTER/HASH, mapping numeric cluster IDs to their hash encoding
   - `variant_counts.tsv` — START/END/MEAN/STDEV of variant counts observed per region

A full chromosome (e.g. chr6) contains one `individual_hashes_*.tsv` / `cluster_hashes_*.tsv` pair per haploblock — around 2,000+ regions for a typical autosome.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# polars >= 1.30.0
# pandas >= 2.0.0
# scikit-learn >= 1.4.0
# scipy >= 1.11.0
# matplotlib >= 3.10.3
# pyarrow >= 15.0.0

import glob
import os

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from botocore import UNSIGNED
from botocore.config import Config
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Location of the S3 bucket for this dataset
bucket = "REPLACE_WITH_ACTUAL_BUCKET"

# Public bucket -- no need to sign requests
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# List the top-level chromosome prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter="/").get("CommonPrefixes", []):
    print(item["Prefix"])

In [ ]:
# Look inside a single chromosome prefix to see the per-region files
for item in s3.list_objects_v2(Bucket=bucket, Prefix="chr21/", MaxKeys=10).get("Contents", []):
    print(item["Key"])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Two formats are used, chosen for different reasons:

- **FASTA** (`_all_seqs.fasta`, `_rep_seq.fasta`) holds the actual haplotype sequences within each haploblock, and cluster representative sequences. FASTA is the standard for sequence data and is well supported by libraries like Biopython or `polars`/`pandas` after light parsing.
- **TSV** (all other files) holds the tabular summaries: haploblock boundaries, cluster assignments, and — the core of this dataset — the binary genomic hashes. Each hash is a fixed-format binary string encoding haplotype-resolved variant co-occurrence within one haploblock (see Kubica et al., 2025 for the exact encoding scheme). TSV keeps this simple, human-readable, and trivially loadable with `polars.read_csv(..., separator="\t")`.

**One practical gotcha worth flagging**: the HASH columns are long strings of `0`s and `1`s (up to ~50 characters). Some CSV parsers will try to infer these as an integer column and overflow, since a 50-digit number is far larger than a standard integer type can hold. Force the column to string type explicitly, e.g. in polars: `pl.read_csv(path, separator="\t", schema_overrides={"HASH": pl.Utf8})`.

`individual_hashes_*.tsv` is haplotype-level, not sample-level: each individual appears as two rows (`_hap0` and `_hap1`). For most downstream analyses you'll want to collapse both haplotypes into one per-sample genotype (e.g. a sorted pair) before treating "sample" as your unit of analysis.

In [ ]:
# Example: load a single region's individual_hashes file directly from S3
key = "chr21/individual_hashes_14284114-14356253.tsv"

obj = s3.get_object(Bucket=bucket, Key=key)
df = pl.read_csv(obj["Body"].read(), separator="\t", schema_overrides={"HASH": pl.Utf8})
df.head()

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

A natural first look at this dataset is the distribution of haploblock lengths across a chromosome. Haploblocks are defined by recombination hotspots, so their lengths reflect real, uneven recombination-rate variation across the genome -- most are short, but a long tail of much longer, more conserved blocks stands out.

In [ ]:
key = "chr6/haploblock_boundaries_chr6.tsv"
obj = s3.get_object(Bucket=bucket, Key=key)
boundaries = pl.read_csv(obj["Body"].read(), separator="\t")

lengths = (boundaries["END"] - boundaries["START"]).to_numpy()

plt.figure(figsize=(12, 7), dpi=100, facecolor="white")
plt.hist(lengths, bins=40, color="#3498db", edgecolor="white", linewidth=1.2, alpha=0.8)
plt.title("Distribution of Haploblock Lengths (chr6)", fontsize=16, pad=20, fontweight="bold")
plt.xlabel("Haploblock length (bp)", fontsize=12, labelpad=10)
plt.ylabel("Count", fontsize=12, labelpad=10)
plt.grid(True, linestyle="--", alpha=0.3, color="gray")

ax = plt.gca()
ax.set_facecolor("#f8f9fa")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Number of haploblocks: {len(lengths)}")
print(f"Median length: {np.median(lengths):.0f} bp")
print(f"Longest block: {lengths.max():.0f} bp")

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question: can an individual's population ancestry be predicted purely from their haploblock cluster assignments, with no other genomic information?**

This is a direct, testable version of a claim from the underlying methodology paper: because haploblock clustering captures population structure (see Table 1 in Kubica et al., 2025, where TNF-alpha and HLA-A cluster counts diverge sharply across CHB/GBR/PUR/26-population comparisons), cluster hashes alone should carry a learnable ancestry signal.

**Approach**: for three 1000 Genomes populations -- CHB (Han Chinese), GBR (British), and PUR (Puerto Rican) -- we pulled each sample's per-haploblock cluster hash across chromosome 6, collapsed each sample's two haplotypes into one diploid genotype label per region, and one-hot encoded the result. Population labels came from the public IGSR sample metadata, not from anything in this dataset itself. A Random Forest was trained to predict population from these encoded haploblock genotypes alone.

Because each region contributes its own set of possible cluster labels, the full feature space is large (169,533 columns for 310 samples across 2,287 chr6 regions) but very sparse -- each sample only has one non-zero entry per region, so a sparse matrix representation keeps this tractable at full-chromosome scale.

In [ ]:
# Cell 1: pull individual_hashes files for many regions from S3 and build a
# long-format (sample, region, genotype) table. Diploid haplotypes (hap0/hap1)
# are collapsed into one sorted genotype label per sample per region.
#
# NOTE: set MAX_REGIONS to a small number (e.g. 50) for a quick interactive
# run. The 97% accuracy result discussed below was obtained with
# MAX_REGIONS=None (all 2,287 chr6 regions), which takes longer to pull and
# process -- shown here capped for a fast, runnable demo.

chrom_prefix = "chr6/"
MAX_REGIONS = 50

keys = [
    obj["Key"]
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=chrom_prefix)
    for obj in page.get("Contents", [])
    if "individual_hashes_" in obj["Key"]
]
keys = sorted(keys)[:MAX_REGIONS] if MAX_REGIONS else sorted(keys)
print(f"Using {len(keys)} region files")

frames = []
for key in keys:
    region = key.split("individual_hashes_")[-1].replace(".tsv", "")
    obj = s3.get_object(Bucket=bucket, Key=key)
    df = pl.read_csv(obj["Body"].read(), separator="\t", schema_overrides={"HASH": pl.Utf8})
    df = df.with_columns([
        pl.col("INDIVIDUAL").str.extract(r"^([^_]+)", 1).alias("sample_id"),
        pl.lit(region).alias("region"),
    ])
    genotype = df.group_by(["sample_id", "region"]).agg(pl.col("HASH").sort())
    frames.append(genotype)

long_df = pl.concat(frames)
print(f"{long_df['region'].n_unique()} regions, {long_df.height} sample-region rows")

In [ ]:
# Cell 2: attach population labels from the public IGSR sample metadata,
# then sparse-encode each region's genotype as one-hot features. Sparse
# encoding keeps memory proportional to actual observations rather than
# samples x total_categories_across_all_regions -- essential once this
# scales to a full chromosome (169,533 columns at genome-wide scale).

igsr_urls = {
    "CHB": "REPLACE_WITH_IGSR_CHB_URL_OR_PATH",
    "GBR": "REPLACE_WITH_IGSR_GBR_URL_OR_PATH",
    "PUR": "REPLACE_WITH_IGSR_PUR_URL_OR_PATH",
}
panel = pd.concat([pd.read_csv(url, sep="\t") for url in igsr_urls.values()], ignore_index=True)
sample_to_pop = panel.set_index("Sample name")["Population code"].to_dict()

wide_df = long_df.to_pandas().pivot(index="sample_id", columns="region", values="HASH")
join_list = lambda h: "|".join(h) if isinstance(h, (list, tuple, np.ndarray)) else h
wide_df = wide_df.apply(lambda col: col.map(join_list))
wide_df["population"] = wide_df.index.map(sample_to_pop)
wide_df = wide_df.dropna(subset=["population"])

region_cols = [c for c in wide_df.columns if c != "population"]
blocks, feature_regions = [], []
for region in region_cols:
    codes, uniques = pd.factorize(wide_df[region])
    block = sparse.csr_matrix(
        (np.ones(len(codes)), (np.arange(len(codes)), codes)),
        shape=(len(codes), len(uniques)),
    )
    blocks.append(block)
    feature_regions.extend([region] * len(uniques))

X = sparse.hstack(blocks, format="csr")
y = wide_df["population"]
print(f"Feature matrix: {X.shape} (sparse, {X.nnz} non-zero entries)")
print(y.value_counts())

In [ ]:
# Cell 3: train a Random Forest and evaluate on a held-out test split.
# Feature importances are aggregated back up to the region level -- a
# data-driven echo of the per-haploblock weight (alpha_ih) in the linear
# model sketched in Kubica et al., 2025.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)

clf = RandomForestClassifier(n_estimators=300, random_state=0)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))

importances = pd.Series(clf.feature_importances_)
region_importance = importances.groupby(feature_regions).sum().sort_values(ascending=False)

region_importance.head(15).plot(kind="barh", figsize=(8, 6))
plt.xlabel("Aggregated feature importance")
plt.title("Most informative haploblock regions for population classification")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

A few additional diagnostic figures make this result easier to read at a glance.

In [ ]:
# Cell 4: confusion matrix -- shows exactly which populations get confused
# with which, not just an aggregate accuracy number.

labels = sorted(y.unique())
cm = confusion_matrix(y_test, clf.predict(X_test), labels=labels)

fig, ax = plt.subplots(figsize=(6, 5.5), dpi=100)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title("Population Classification -- Confusion Matrix", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: 2D projection of samples colored by population, so the structure
# the classifier is exploiting can be seen directly rather than taken on
# faith. Uses TruncatedSVD (works natively on sparse matrices, unlike PCA).
#
# Note: with 169k+ sparse features, the % variance captured by just 2
# components is tiny by construction -- these axes are simply the best 2D
# separating directions found, not a variance-proportion summary.

svd = TruncatedSVD(n_components=2, random_state=0)
coords = svd.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
colors = {"CHB": "#e74c3c", "GBR": "#3498db", "PUR": "#2ecc71"}
for pop in labels:
    mask = (y == pop).to_numpy()
    ax.scatter(coords[mask, 0], coords[mask, 1], label=pop, alpha=0.7, s=50,
               color=colors.get(pop, "gray"), edgecolor="white", linewidth=0.5)

ax.set_xlabel("SVD Component 1", fontsize=12)
ax.set_ylabel("SVD Component 2", fontsize=12)
ax.set_title("Samples Projected by Haploblock Cluster Assignment", fontsize=13, fontweight="bold", pad=15)
ax.legend(title="Population")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: accuracy vs. number of regions used. Reuses the already-downloaded
# region data from Cell 1/2 -- no additional S3 calls -- by re-encoding
# increasing subsets of the region columns already pivoted into wide_df.

def encode_subset(wide_df, regions_subset):
    blocks = []
    for region in regions_subset:
        codes, uniques = pd.factorize(wide_df[region])
        block = sparse.csr_matrix(
            (np.ones(len(codes)), (np.arange(len(codes)), codes)),
            shape=(len(codes), len(uniques)),
        )
        blocks.append(block)
    return sparse.hstack(blocks, format="csr")

checkpoints = sorted(set([5, 10, 25, len(region_cols)]))
accuracies = []
for n in checkpoints:
    X_sub = encode_subset(wide_df, region_cols[:n])
    Xtr, Xte, ytr, yte = train_test_split(X_sub, y, test_size=0.25, stratify=y, random_state=0)
    clf_sub = RandomForestClassifier(n_estimators=300, random_state=0)
    clf_sub.fit(Xtr, ytr)
    accuracies.append(accuracy_score(yte, clf_sub.predict(Xte)))

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=100)
ax.plot(checkpoints, accuracies, marker="o", color="#3498db", linewidth=2, markersize=7)
ax.axhline(1 / 3, color="gray", linestyle="--", linewidth=1, label="Chance level (33%)")
ax.set_xlabel("Number of haploblock regions used", fontsize=12)
ax.set_ylabel("Held-out accuracy", fontsize=12)
ax.set_title("Classification Accuracy vs. Genomic Coverage (this MAX_REGIONS-capped run)", fontsize=12, fontweight="bold", pad=15)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

**Result**: with `MAX_REGIONS` capped at 50 for a fast interactive run, this reaches roughly 69% held-out accuracy on the three-way CHB/GBR/PUR classification (chance level is 33%). Running the same pipeline across the **full chromosome** (all 2,287 chr6 regions, `MAX_REGIONS=None`) increases this to **97% held-out accuracy**:

```
              precision    recall  f1-score   support
         CHB       1.00      1.00      1.00        27
         GBR       0.93      1.00      0.96        25
         PUR       1.00      0.92      0.96        26
    accuracy                           0.97        78
```

CHB (Han Chinese) separates perfectly from the other two populations. The small remaining confusion is specifically between GBR and PUR -- consistent with known population genetics, since Puerto Rican ancestry includes substantial European admixture, so GBR and PUR haploblock cluster patterns genuinely overlap more with each other than either does with CHB. This is a real, checkable biological pattern, not noise from the classifier.

The jump from 69% (50 regions) to 97% (2,287 regions) demonstrates the paper's core argument directly: population-discriminating signal is distributed across many independent recombination-defined regions, so genome-wide analysis -- not a handful of loci -- is what's needed to reliably resolve fine-grained genotype-phenotype (or here, genotype-ancestry) structure.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

The demo above uses population ancestry as a stand-in phenotype, since it's public and requires no additional data linkage. The genuinely open question is whether the same haploblock-hash representation can predict **real clinical or quantitative phenotypes** (e.g. from a biobank) rather than ancestry alone -- this is the direction the underlying methodology paper proposes for future work (replacing the simple linear model with something like XGBoost to capture nonlinear haploblock interactions, and incorporating diploidy properly rather than collapsing haplotype pairs as done here).

For someone extending this: the sparse, per-region one-hot representation used above scales to a full chromosome without difficulty, but genome-wide (all chromosomes combined) will multiply the feature count substantially further. Consider either (a) a sparse-native model that never materializes a dense array at any point, or (b) a feature-selection or dimensionality-reduction step per chromosome before combining across the genome. Linking this dataset against real phenotype data will also require careful attention to sample-ID harmonization and any relevant data-use restrictions of the phenotype source, which are separate from (and likely more restrictive than) the public, unrestricted nature of this dataset itself.

# DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS BEFORE COMMITTING TO YOUR GITHUB REPOSITORY